In [0]:
spark.sql("USE CATALOG e_comm")
spark.sql("USE SCHEMA bronze")

In [0]:
from pyspark.sql.functions import *
import pandas as pd
import requests
from io import StringIO
from delta.tables import DeltaTable

In [0]:
spark.sql("create table if not exists e_comm.bronze.order_items(order_id string, order_item_id bigint, product_id string, seller_id string, shipping_limit_date string, price double, freight_value double, merge_flag boolean, ingestion_ts timestamp) using delta ")

In [0]:
from pyspark.sql.functions import lit, current_timestamp
import requests
import pandas as pd
from io import StringIO
url = "https://raw.githubusercontent.com/deepakmali17/E_commerce_dataplatform/refs/heads/main/datasets/order_items.csv"

response = requests.get(url)

response.raise_for_status()
pdf  = pd.read_csv(StringIO(response.text))
df = spark.createDataFrame(pdf)
df_source = df.withColumn("merge_flag", lit(False)).withColumn("ingestion_ts", current_timestamp())


df_target = DeltaTable.forName(spark, "e_comm.bronze.order_items")
df_target.alias("t").merge(df_source.alias("s"), "t.order_id = s.order_id and t.order_item_id = s.order_item_id")\
    .whenMatchedUpdateAll(
        condition = "t.product_id <> s.product_id or t.seller_id <> s.seller_id or t.shipping_limit_date <> s.shipping_limit_date or t.price <> s.price or t.freight_value <> s.freight_value")\
        .whenNotMatchedInsertAll().execute()



In [0]:
%sql
select * from e_comm.bronze.order_items